In [1]:
import os
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
import optuna
import warnings
import logging
import xgboost as xgb
# Used for debugging
print(xgb.__file__)
print(type(xgb))
print(dir(xgb)[:50])
warnings.filterwarnings('ignore', category=FutureWarning)
optuna.logging.set_verbosity(logging.WARNING)

print("Loading and filtering data...")
data_dir = '../preprocesing_data/processed_csv'
sales_path = os.path.join(data_dir, 'sales_data_preprocessed.csv')
weather_path = os.path.join(data_dir, 'weather_data_hourly.csv')
holidays_path = os.path.join(data_dir, 'holidays_data_preprocessed.csv')
events_path = os.path.join(data_dir, 'aberdeen_events_master_timeline.csv')

# ==========================================
# Data Ingestion: Sales (Global Melt)
# ==========================================
sales_df = pd.read_csv(sales_path)
sales_df['Date'] = pd.to_datetime(sales_df['Date']).dt.normalize()
if 'Time' in sales_df.columns:
    sales_df = sales_df.drop(columns=['Time'])
daily_sales = sales_df.groupby('Date').sum(numeric_only=True).reset_index()

# Extract all product columns (everything except Date)
product_cols = [c for c in daily_sales.columns if c != 'Date']

# MELT: Convert from wide to long
df_long = pd.melt(daily_sales, id_vars=['Date'], value_vars=product_cols,
                  var_name='Product_Name', value_name='Sales')
df_long['Product_Name'] = df_long['Product_Name'].astype('category')

# Only keep my 64 target products

# ==========================================
# PRODUCTS TO FORECAST
# ==========================================
# These are the 64 specific menu items I want to predict
PRODUCTS_TO_FORECAST = [
 'Avo & Hal Muffin',
 'Avo, Egg & Bacon',
 'Avo, Feta & Tom',
 'Avocado on Toast',
 'Bacon',
 'Bacon Bap',
 'Bacon Egg Brioch',
 'Bacon Waffle',
 'Baked Beans',
 'Baked Beans JP',
 'Bean Soldiers',
 'Big Breakfast',
 'Black Pudding',
 'Breakfast Hash',
 'Breakfast Muffin',
 'Breakfast Wrap',
 'Buttd Mushrooms',
 'Cheese & Bean JP',
 'Cheese JP',
 'Chick Flatbread',
 'Chicken Club',
 'Chilli Carne JP',
 'Egg Bacon Brioch',
 'Egg Bap',
 'Extra Beans',
 'F.Eggs on Toast',
 'Festive Stack',
 'Fried Egg',
 'Hash Brown',
 'Hash Brown Bites',
 'Little Avo Toast',
 'Little Bean Toas',
 'Little Egg Toast',
 'Ltle Bfast Bacon',
 'Ltle Bfast Saus',
 'Mini Hash Browns',
 'P.Eggs on Toast',
 'Poached Egg',
 'Posh Beans',
 'Roll & Butter ',
 'S.Eggs on Toast',
 'Sausage',
 'Sausage Bap',
 'Scrambled Egg',
 'Streaky Bacon',
 'Tattie Scone',
 'The Breakfast',
 'Toasted Teacake',
 'Tuna JP',
 'Tuna Mayo Mix',
 'Tuna Melt Panini',
 'Tuna Panini',
 'Tuna Toastie',
 'Veg Sausage Bap',
 'Vegan Breakfast',
 'Vegan Sausage',
 'Veggie Bap',
 'Veggie Breakfast',
 'Bakery',
 'White Toast Bread',
 'Brown Toast Bread',
 'Porridge',
 'Sourdough Toast Bread',
 'Multiseed Toast Bread',
]
df_long = df_long[df_long['Product_Name'].isin(PRODUCTS_TO_FORECAST)].reset_index(drop=True)

# ==========================================
# Data Ingestion: Weather
# ==========================================
weather_df = pd.read_csv(weather_path)
weather_df['Date'] = pd.to_datetime(weather_df['Date']).dt.normalize()
if 'weather_code' in weather_df.columns:
    weather_df['is_clear'] = (weather_df['weather_code'] == 0).astype(int)
    weather_df['is_cloudy'] = weather_df['weather_code'].isin([1, 2, 3, 45, 48]).astype(int)
    weather_df['is_rain'] = weather_df['weather_code'].isin([51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82, 95, 96, 99]).astype(int)
    weather_df['is_snow'] = weather_df['weather_code'].isin([71, 73, 75, 77, 85, 86]).astype(int)
    weather_df = weather_df.drop(columns=['weather_code'])
daily_weather = weather_df.groupby('Date').max().reset_index()
if 'Time' in daily_weather.columns:
    daily_weather = daily_weather.drop(columns=['Time'])

# ==========================================
# Data Ingestion: Holidays & Events
# ==========================================
holidays_df = pd.read_csv(holidays_path)
holidays_df['Date'] = pd.to_datetime(holidays_df['Date']).dt.normalize()
daily_holidays = holidays_df.groupby('Date').max().reset_index()

events_df = pd.read_csv(events_path)
events_df['Date'] = pd.to_datetime(events_df['Date']).dt.normalize()
daily_events = events_df.groupby('Date').max(numeric_only=True).reset_index()


/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/home/alex/miniconda3/envs/jupyterproject-project/lib/python3.9/site-packages/xgboost/__init__.py
<class 'module'>
['Booster', 'DMatrix', 'DataIter', 'DeviceQuantileDMatrix', 'QuantileDMatrix', 'RabitTracker', 'XGBClassifier', 'XGBModel', 'XGBRFClassifier', 'XGBRFRegressor', 'XGBRanker', 'XGBRegressor', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_py_version', '_typing', 'build_info', 'callback', 'collective', 'compat', 'config', 'config_context', 'core', 'cv', 'dask', 'data', 'get_config', 'libpath', 'plot_importance', 'plot_tree', 'plotting', 'set_config', 'sklearn', 'to_graphviz', 'tracker', 'train', 'training']
Loading and filtering data...


In [2]:
# ==========================================
# Feature Engineering: Merge & Lags
# ==========================================
# Legacy code, just in case i will need it
# df_long['day_sin'] = np.sin(2 * np.pi * df_long['Date'].dt.dayofweek / 7)
# df_long['day_cos'] = np.cos(2 * np.pi * df_long['Date'].dt.dayofweek / 7)
# df_long['month_sin'] = np.sin(2 * np.pi * (df_long['Date'].dt.month - 1) / 12)
# df_long['month_cos'] = np.cos(2 * np.pi * (df_long['Date'].dt.month - 1) / 12)
# df_long['Is_Weekend'] = df_long['Date'].dt.dayofweek.isin([5, 6]).astype(int)

df = df_long.merge(daily_weather, on='Date', how='left')
df = df.merge(daily_holidays, on='Date', how='left')
df = df.merge(daily_events, on='Date', how='left')

# Drop problematic object columns if they exist after merge
for col in ['date', 'Time']:
    if col in df.columns:
        df = df.drop(columns=[col])

# Exclude categorical columns from fillna(0)
categorical_cols = df.select_dtypes(include=['category']).columns
non_categorical_cols = df.columns.difference(categorical_cols)
df[non_categorical_cols] = df[non_categorical_cols].fillna(0)

# Add Base Lags (GROUPED BY PRODUCT)
df = df.sort_values(['Product_Name', 'Date']).reset_index(drop=True)
df['sales_1_step_ago'] = df.groupby('Product_Name', observed=False)['Sales'].shift(1)
df['sales_7_steps_ago'] = df.groupby('Product_Name', observed=False)['Sales'].shift(7)
df['sales_30_steps_ago'] = df.groupby('Product_Name', observed=False)['Sales'].shift(30)
df = df.dropna().reset_index(drop=True)

# Ensure no negative sales in training data
df['Sales'] = df['Sales'].clip(lower=0)

feature_cols = [c for c in df.columns if c not in ['Date', 'Sales']]


In [3]:
# ==========================================
# Model Training: Split & Tune
# ==========================================
train_end = pd.to_datetime('2025-10-01')
val_end = pd.to_datetime('2025-11-01')
test_end = pd.to_datetime('2025-11-30')

train_df = df[df['Date'] <= train_end].copy()
val_df = df[(df['Date'] > train_end) & (df['Date'] <= val_end)].copy()
test_df = df[(df['Date'] > val_end) & (df['Date'] <= test_end)].copy().reset_index(drop=True)

X_train, y_train = train_df[feature_cols], train_df['Sales']
X_val, y_val = val_df[feature_cols], val_df['Sales']

print("\nRunning Optuna Hyperparameter Tuning on Entire Catalog (30 Trials)...")
def objective(trial):
    param = {
        "n_estimators": 1000,
        "early_stopping_rounds": 50,
        "learning_rate": trial.suggest_float("learning_rate", 1e-4, 0.1, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 10),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 1e-8, 1.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "enable_categorical": True,
        "tree_method": "hist"
    }

    model = xgb.XGBRegressor(**param)
    model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_val, y_val)], verbose=False)
    preds = model.predict(X_val)
    return np.sqrt(mean_squared_error(y_val, preds))

study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=30)
best_params = study.best_params
best_params.update({
    "n_estimators": 1000,
    "early_stopping_rounds": 30,
    "enable_categorical": True,
    "tree_method": "hist"
})

print("Training final global model...")
model = xgb.XGBRegressor(**best_params)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)



Running Optuna Hyperparameter Tuning on Entire Catalog (30 Trials)...
Training final global model...


XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.793799850883132, device=None,
             early_stopping_rounds=30, enable_categorical=True,
             eval_metric=None, feature_types=None, gamma=0.04482830018326405,
             grow_policy=None, importance_type=None,
             interaction_constraints=None, learning_rate=0.028394731176148106,
             max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
             max_delta_step=None, max_depth=7, max_leaves=None,
             min_child_weight=8, missing=nan, monotone_constraints=None,
             multi_strategy=None, n_estimators=1000, n_jobs=None,
             num_parallel_tree=None, random_state=None, ...)

In [4]:
# ==========================================
# Forecasting: Recursive Loop
# ==========================================
print("\nRunning True Recursive Forecast for ALL Products...")

# Map of which lag corresponds to which column
LAG_MAP = {1: 'sales_1_step_ago', 7: 'sales_7_steps_ago', 30: 'sales_30_steps_ago'}

all_products = test_df['Product_Name'].cat.categories.tolist()
test_dates = sorted(test_df['Date'].unique())

# Set up test dataframe for predictions
test_df['XGB_Prediction'] = 0.0
test_df['original_sales_1_step_ago'] = test_df['sales_1_step_ago'].copy()

# Build a lookup dictionary for blazing fast row finding: (product, date) -> index
idx_lookup = {}
for idx, row in test_df.iterrows():
    idx_lookup[(row['Product_Name'], row['Date'])] = idx

# Process day by day, all products simultaneously
for day_i, current_date in enumerate(test_dates):
    # Grab all product rows for exactly this date
    day_mask = test_df['Date'] == current_date
    day_indices = test_df.index[day_mask].tolist()

    if len(day_indices) == 0:
        continue

    # STEP 1: Predict all products for today at once
    X_day = test_df.loc[day_indices, feature_cols]
    preds = model.predict(X_day)

    # STEP 2: Clip predictions so XGBoost never predicts negative sales
    preds = np.clip(preds, 0, None)

    # STEP 3: Save predictions to the scorecard
    test_df.loc[day_indices, 'XGB_Prediction'] = preds

    # STEP 4: Feed predictions forward into the exact lag columns for the future dates
    for row_idx, pred_val in zip(day_indices, preds):
        product = test_df.at[row_idx, 'Product_Name']

        for lag_days, lag_col in LAG_MAP.items():
            future_date = current_date + pd.Timedelta(days=lag_days)
            future_key = (product, future_date)

            # If the future date exists in our test set, overwrite the lag with our prediction
            if future_key in idx_lookup:
                future_idx = idx_lookup[future_key]
                test_df.at[future_idx, lag_col] = pred_val

print("Recursive forecast complete for all products.")



Running True Recursive Forecast for ALL Products...
Recursive forecast complete for all products.


In [5]:
# ==========================================
# Evaluation: Model Scorecard
# ==========================================
# ==========================================
# North Star Metrics
# ==========================================
# Focus on WAPE for total inventory accuracy and MASE to ensure the model outperforms 
# the naive lag-1 baseline. MAE provides unit-level error for kitchen staff, 
# while Bias tracks over/under-prediction trends to manage waste and stockouts.

def calculate_north_star_metrics(evaluation_slice):

    if len(evaluation_slice) == 0:
        return {m: 0 for m in ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE']}

    actual_sales = evaluation_slice['Sales'].values.astype(float)
    predicted_sales = evaluation_slice['XGB_Prediction'].values.astype(float)
    # Use 'original_sales_1_step_ago' to compare against the real past sales, not recursively predicted ones
    naive_baseline = evaluation_slice['original_sales_1_step_ago'].values.astype(float)

    #  NORTH STAR 1: WAPE (Weighted Absolute Percentage Error)
    # Sum of absolute errors / Sum of actual sales
    # This tells me the total % of inventory I got wrong across all products
    total_absolute_errors = np.sum(np.abs(actual_sales - predicted_sales))
    total_actual_sales = np.sum(np.abs(actual_sales))
    wape = total_absolute_errors / total_actual_sales if total_actual_sales > 0 else np.nan

    #  NORTH STAR 2: MASE (Mean Absolute Scaled Error)
    # My model's MAE divided by naive baseline MAE
    # < 1.0 means my model adds value, > 1.0 means I should just use yesterday's number
    mae = mean_absolute_error(actual_sales, predicted_sales)
    naive_baseline_mae = mean_absolute_error(actual_sales, naive_baseline)
    mase = mae / naive_baseline_mae if naive_baseline_mae > 0 else np.nan

    #  NORTH STAR 3: MAE (Mean Absolute Error in units)
    # The kitchen needs to know: "how many units off will I typically be?"
    # (mae already calculated above)

    #  NORTH STAR 4: Bias (directional error)
    # Negative = I'm under-predicting (we'll run out of stock)
    # Positive = I'm over-predicting (we'll waste food)
    forecast_bias = np.mean(predicted_sales - actual_sales)

    # Legacy metrics (keeping for backwards compatibility)
    mse = mean_squared_error(actual_sales, predicted_sales)
    rmse = np.sqrt(mse)

    # MAPE — only on non-zero actuals to avoid division by zero
    nonzero_mask = actual_sales > 0
    mape = mean_absolute_percentage_error(actual_sales[nonzero_mask], predicted_sales[nonzero_mask]) if nonzero_mask.sum() > 0 else np.nan

    # SMAPE — symmetric version, less sensitive to zeros
    abs_errors = np.abs(predicted_sales - actual_sales)
    abs_denominator = (np.abs(actual_sales) + np.abs(predicted_sales)) / 2.0
    valid_mask = abs_denominator > 0
    smape = np.mean(abs_errors[valid_mask] / abs_denominator[valid_mask]) if valid_mask.sum() > 0 else np.nan

    return {'WAPE': wape, 'MASE': mase, 'MAE': mae, 'Bias': forecast_bias,
            'MSE': mse, 'RMSE': rmse, 'MAPE': mape, 'SMAPE': smape}


    y_true = df_slice['Sales'].values.astype(float)
    y_pred = df_slice['XGB_Prediction'].values.astype(float)
    y_naive = df_slice['original_sales_1_step_ago'].values.astype(float)

    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)

    # Guard against zero sales for MAPE
    nonzero_mask = y_true > 0
    if nonzero_mask.sum() > 0:
        mape = mean_absolute_percentage_error(y_true[nonzero_mask], y_pred[nonzero_mask])
    else:
        mape = np.nan

    # SMAPE with zero guard
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2.0
    valid = denominator > 0
    if valid.sum() > 0:
        smape = np.mean(numerator[valid] / denominator[valid])
    else:
        smape = np.nan

    naive_mae = mean_absolute_error(y_true, y_naive)
    mase = mae / naive_mae if naive_mae > 0 else np.nan

    forecast_bias = np.mean(y_pred - y_true)

    return {'MAE': mae, 'MSE': mse, 'RMSE': rmse, 'MAPE': mape,
            'SMAPE': smape, 'MASE': mase, 'Bias': forecast_bias}

# Bacon Bap detailed report
target_product = "Bacon Bap"
target_test_df = test_df[test_df['Product_Name'] == target_product].copy()
target_test_df = target_test_df.sort_values('Date').reset_index(drop=True)
target_test_df['Date_Only'] = target_test_df['Date'].dt.date

target_date_1 = pd.to_datetime('2025-11-02').date()
target_date_7 = pd.to_datetime('2025-11-08').date()
target_date_30 = pd.to_datetime('2025-11-30').date()

df_1_day = target_test_df[target_test_df['Date_Only'] == target_date_1]
df_1_week = target_test_df[(target_test_df['Date_Only'] >= target_date_1) & (target_test_df['Date_Only'] <= target_date_7)]
df_1_month = target_test_df[(target_test_df['Date_Only'] >= target_date_1) & (target_test_df['Date_Only'] <= target_date_30)]

comparison_df_bacon = pd.DataFrame({
    'Metric': ['WAPE', 'MASE', 'MAE', 'Bias', 'MSE', 'RMSE', 'MAPE', 'SMAPE'],
    '1-Day (Nov 2)': list(calculate_north_star_metrics(df_1_day).values()),
    '1-Week (Nov 2-8)': list(calculate_north_star_metrics(df_1_week).values()),
    '1-Month (Nov 2-30)': list(calculate_north_star_metrics(df_1_month).values())
})

for col in comparison_df_bacon.columns[1:]:
    comparison_df_bacon[col] = comparison_df_bacon[col].apply(
        lambda x: f"{float(x):.4f}" if not np.isnan(x) else "N/A"
    )

print("\n======================================================")
print(f"  RECURSIVE XGBOOST METRICS (BLIND TEST): {target_product}")
print("======================================================")
print(comparison_df_bacon.to_string(index=False))

target_test_df['XGB_Prediction_Rounded'] = target_test_df['XGB_Prediction'].round().astype(int)
target_test_df['Mistake_Amount'] = (target_test_df['Sales'] - target_test_df['XGB_Prediction_Rounded']).abs()

print("\n==================================================")
print(f"  RECURSIVE XGBOOST REALITY CHECK: {target_product}")
print("==================================================")
print(target_test_df[['Date_Only', 'Sales', 'XGB_Prediction_Rounded', 'Mistake_Amount']].head(10).to_string(index=False))

#  Aggregate metrics across ALL products
print("\n==================================================")
print("  PER-PRODUCT 1-MONTH METRICS (Nov 2-30)")
print("==================================================")

product_results = []
for product in PRODUCTS_TO_FORECAST:
    p_df = test_df[test_df['Product_Name'] == product].copy()
    p_df['Date_Only'] = p_df['Date'].dt.date
    p_month = p_df[(p_df['Date_Only'] >= target_date_1) & (p_df['Date_Only'] <= target_date_30)]
    metrics = calculate_north_star_metrics(p_month)
    metrics['Product'] = product
    product_results.append(metrics)

product_leaderboard_df = pd.DataFrame(product_results)
product_leaderboard_df = product_leaderboard_df[['Product', 'MAE', 'RMSE', 'MAPE', 'MASE', 'Bias']]
product_leaderboard_df = product_leaderboard_df.sort_values('MAE')

for col in ['MAE', 'RMSE', 'MAPE', 'MASE', 'Bias']:
    product_leaderboard_df[col] = product_leaderboard_df[col].apply(
        lambda x: f"{x:.4f}" if not np.isnan(x) else "N/A"
    )

print(product_leaderboard_df.head(15).to_string(index=False))
print("... (showing top 15 by MAE)")

#  Overall aggregate
all_month = test_df.copy()
all_month['Date_Only'] = all_month['Date'].dt.date
all_month = all_month[(all_month['Date_Only'] >= target_date_1) & (all_month['Date_Only'] <= target_date_30)]
overall = calculate_north_star_metrics(all_month)

print("\n==================================================")
print("  OVERALL AGGREGATE METRICS (All Products, Nov 2-30)")
print("==================================================")
for k, v in overall.items():
    print(f"  {k:>6s}: {v:.4f}" if not np.isnan(v) else f"  {k:>6s}: N/A")

#  Feature importance
print("\n==================================================")
print("  TOP FEATURE IMPORTANCES (gain)")
print("==================================================")
importance = model.get_booster().get_score(importance_type='gain')
importance_sorted = sorted(importance.items(), key=lambda x: x[1], reverse=True)[:10]
for feat, score in importance_sorted:
    print(f"  {feat:<30s} {score:.1f}")



  RECURSIVE XGBOOST METRICS (BLIND TEST): Bacon Bap
Metric 1-Day (Nov 2) 1-Week (Nov 2-8) 1-Month (Nov 2-30)
  WAPE        0.3623           0.2879             0.2768
  MASE           N/A           0.9243             0.7542
   MAE        9.4203           5.0176             4.0571
  Bias       -9.4203          -3.2028            -0.8398
   MSE       88.7423          36.2283            27.6069
  RMSE        9.4203           6.0190             5.2542
  MAPE        0.3623           0.3055             0.3148
 SMAPE        0.4425           0.3105             0.2830

  RECURSIVE XGBOOST REALITY CHECK: Bacon Bap
 Date_Only  Sales  XGB_Prediction_Rounded  Mistake_Amount
2025-11-02     26                      17               9
2025-11-03      8                      14               6
2025-11-04     14                      13               1
2025-11-05     14                      14               0
2025-11-06     22                      13               9
2025-11-07     18                      1

In [6]:
# ==========================================
# Persistence: SQLite Tracking
# ==========================================
# Use SQLite to maintain a historical record of model performance, 
# allowing for trend analysis in Tableau/PowerBI for the kitchen manager.

import sqlite3
from datetime import datetime

os.makedirs('../results', exist_ok=True)

# Generate a unique run ID so I can track this specific model run
prediction_timestamp = datetime.now()
run_id = prediction_timestamp.strftime('%Y%m%d_%H%M') + '_XGBoost_Simple_Daily'

print(f"Saving results for run: {run_id}")

# 1. Still save CSVs for quick access 
comparison_df_bacon.to_csv('../results/xgboost_simple_daily_recursive_results.csv', index=False)
product_leaderboard_df.to_csv('../results/xgboost_simple_daily_per_product_results.csv', index=False)
test_df[['Date', 'Product_Name', 'Sales', 'XGB_Prediction']].to_csv(
    '../results/xgboost_simple_daily_all_predictions.csv', index=False)
print("  CSVs saved to ../results/")

#  2. SQLite Storage Framework 
# Connect to (or create) my model tracking database
database_path = '../results/model_tracking.db'
db_connection = sqlite3.connect(database_path)

with db_connection:
    # Create tables if they don't exist yet
    # predictions_log: stores every single prediction for charting actual vs predicted
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS predictions_log (
            run_id TEXT,
            model_type TEXT,
            target_date TEXT,
            prediction_made_on TEXT,
            product_name TEXT,
            actual_sales REAL,
            predicted_sales REAL
        )
    """)

    # metrics_summary: stores aggregated North Star metrics per run/product/horizon
    db_connection.execute("""
        CREATE TABLE IF NOT EXISTS metrics_summary (
            run_id TEXT,
            model_type TEXT,
            product_name TEXT,
            evaluation_horizon TEXT,
            WAPE REAL,
            MASE REAL,
            MAE REAL,
            Bias REAL
        )
    """)

    # Populate predictions_log 
    predictions_for_db = test_df[['Date', 'Product_Name', 'Sales', 'XGB_Prediction']].copy()
    predictions_for_db = predictions_for_db.rename(columns={
        'Date': 'target_date', 'Product_Name': 'product_name',
        'Sales': 'actual_sales', 'XGB_Prediction': 'predicted_sales'
    })
    predictions_for_db['run_id'] = run_id
    predictions_for_db['model_type'] = 'XGBoost_Simple_Daily'
    predictions_for_db['prediction_made_on'] = str(prediction_timestamp)
    predictions_for_db['target_date'] = predictions_for_db['target_date'].astype(str)
    predictions_for_db.to_sql('predictions_log', db_connection, if_exists='append', index=False)
    print(f"  Logged {len(predictions_for_db)} predictions to predictions_log")

    #  Populate metrics_summary 
    evaluation_start = pd.to_datetime('2025-11-02').date()
    evaluation_week_end = pd.to_datetime('2025-11-08').date()
    evaluation_month_end = pd.to_datetime('2025-11-30').date()

    test_df['Date_Only'] = test_df['Date'].dt.date
    month_data = test_df[(test_df['Date_Only'] >= evaluation_start) & (test_df['Date_Only'] <= evaluation_month_end)]
    week_data = month_data[month_data['Date_Only'] <= evaluation_week_end]
    day_data = month_data[month_data['Date_Only'] == evaluation_start]
    horizons = {'1-Day': day_data, '1-Week': week_data, '1-Month': month_data}

    metrics_rows = []
    for product in all_products:
        for horizon_label, horizon_df in horizons.items():
            product_horizon = horizon_df[horizon_df['Product_Name'] == product]
            if len(product_horizon) > 0:
                m = calculate_north_star_metrics(product_horizon)
                metrics_rows.append({
                    'run_id': run_id, 'model_type': 'XGBoost_Simple_Daily',
                    'product_name': product, 'evaluation_horizon': horizon_label,
                    'WAPE': m['WAPE'], 'MASE': m['MASE'], 'MAE': m['MAE'], 'Bias': m['Bias']
                })

    # Products aggregate row
    for horizon_label, horizon_df in horizons.items():
        if len(horizon_df) > 0:
            m = calculate_north_star_metrics(horizon_df)
            metrics_rows.append({
                'run_id': run_id, 'model_type': 'XGBoost_Simple_Daily',
                'product_name': 'ALL_PRODUCTS', 'evaluation_horizon': horizon_label,
                'WAPE': m['WAPE'], 'MASE': m['MASE'], 'MAE': m['MAE'], 'Bias': m['Bias']
            })

    if metrics_rows:
        metrics_summary_df = pd.DataFrame(metrics_rows)
        metrics_summary_df.to_sql('metrics_summary', db_connection, if_exists='append', index=False)
        print(f"  Logged {len(metrics_summary_df)} metric rows to metrics_summary")

db_connection.close()
print(f"  SQLite database saved to {database_path}")
print(f"  Run ID: {run_id}")
print("\nDone!")


Saving results for run: 20260422_1233_XGBoost_Simple_Daily
  CSVs saved to ../results/
  Logged 1856 predictions to predictions_log
  Logged 195 metric rows to metrics_summary
  SQLite database saved to ../results/model_tracking.db
  Run ID: 20260422_1233_XGBoost_Simple_Daily

Done!
